## TON IoT Data

In [75]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# import pandas as pd
# import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report
import matplotlib.pyplot as plt
# import torch.nn as nn
# import torch.nn.functional as F
# import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import math, time, os, datetime, psutil, gc
import random
import kagglehub
import warnings

%load_ext autoreload
%autoreload 2
from utility import *
from informer import *
from libs import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data Ingestion Step

In [76]:
file='../data/TON_IoT/train_test_network.csv'

In [77]:
### IOT Network
df= pd.read_csv(file)
df.head()
col_resp='type'

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [78]:
df['type'].describe()

count     211043
unique        10
top       normal
freq       50000
Name: type, dtype: object

In [79]:
df.replace('-', 'Missing', inplace=True)
df.fillna('Missing', inplace=True)

X = df.drop(columns=['label', 'type'])
y = df['type']

# Encode the target variable (type)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_

# Encode categorical features
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype(str)
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

In [80]:
# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [81]:
np.unique_counts(y_train)

UniqueCountsResult(values=array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), counts=array([16000, 16000, 16000, 16000,   834, 40000, 16000, 16000, 16000,
       16000]))

In [82]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 4. Build Deep Learning Model
# ==========================================
class DeepClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) 
            # Note: No Softmax here because CrossEntropyLoss includes it
        )

    def forward(self, x):
        return self.network(x)

input_dim = X_train_scaled.shape[1]
num_classes = len(class_names)
model = DeepClassifier(input_dim, num_classes).to(device)

# Define Optimizer and Loss
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss() # Equivalent to sparse_categorical_crossentropy

In [83]:
# ==========================================
# 5. Train the Model
# ==========================================
# Prepare DataLoaders (Equivalent to validation_split and batch_size)
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_subset, val_subset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=512)

epochs = 20
print("\nTraining the model...")

for epoch in range(epochs):
    _=model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Simple Validation check
    _=model.eval()
    val_loss = 0
    with torch.no_grad():
        for val_X, val_y in val_loader:
            val_X, val_y = val_X.to(device), val_y.to(device)
            val_loss += criterion(model(val_X), val_y).item()
            
    # print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss/len(train_loader):.4f} - Val Loss: {val_loss/len(val_loader):.4f}")



Training the model...


In [84]:
# ==========================================
# 6. Evaluate & Print Metrics
# ==========================================
print("\nEvaluating the model on test data...")
_=model.eval()
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)

with torch.no_grad():
    y_pred_logits = model(X_test_tensor)
    y_pred = torch.argmax(y_pred_logits, dim=1).cpu().numpy()

accuracy = accuracy_score(y_test, y_pred)
print(f"\n---> Test Accuracy: {accuracy * 100:.2f}% <---")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))


Evaluating the model on test data...

---> Test Accuracy: 97.26% <---

Classification Report:
              precision    recall  f1-score   support

    backdoor       1.00      1.00      1.00      4000
        ddos       0.98      0.97      0.97      4000
         dos       0.97      0.97      0.97      4000
   injection       0.92      0.90      0.91      4000
        mitm       0.80      0.77      0.78       209
      normal       1.00      1.00      1.00     10000
    password       0.91      0.94      0.92      4000
  ransomware       0.99      1.00      1.00      4000
    scanning       0.97      0.96      0.96      4000
         xss       0.99      0.99      0.99      4000

    accuracy                           0.97     42209
   macro avg       0.95      0.95      0.95     42209
weighted avg       0.97      0.97      0.97     42209



# Informer

In [85]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df[col_resp] = df[col_resp].astype(str).str.strip()
# df[col_resp] = le.fit_transform(df[col_resp])
# df[col_resp].value_counts()

if(len(np.array(df[col_resp].value_counts()))>2):
    # 1. Get unique types and remove 'normal'
    attack_types = [t for t in df[col_resp].unique() if t != 'normal']

    # 2. Create a mapping where 'normal' is always 0
    mapping = {t: i+1 for i, t in enumerate(attack_types)}
    mapping['normal'] = 0

    # 3. Apply the mapping
    df[col_resp] = df[col_resp].map(mapping)

    print(f"Manual Mapping: {mapping}")

Manual Mapping: {'backdoor': 1, 'ddos': 2, 'dos': 3, 'injection': 4, 'mitm': 5, 'password': 6, 'ransomware': 7, 'scanning': 8, 'xss': 9, 'normal': 0}


In [86]:
X.columns
y.value_counts()

Index(['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service',
       'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes',
       'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query',
       'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA',
       'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed',
       'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth',
       'http_method', 'http_uri', 'http_version', 'http_request_body_len',
       'http_response_body_len', 'http_status_code', 'http_user_agent',
       'http_orig_mime_types', 'http_resp_mime_types', 'weird_name',
       'weird_addl', 'weird_notice'],
      dtype='object')

type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
Name: count, dtype: int64

## Data Preprocessing

In [87]:
df

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,Missing,290.371539,101568,2592,OTH,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,1
1,192.168.1.193,49180,192.168.1.37,8080,tcp,Missing,0.000102,0,0,REJ,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,1
2,192.168.1.193,49180,192.168.1.37,8080,tcp,Missing,0.000148,0,0,REJ,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,1
3,192.168.1.193,49180,192.168.1.37,8080,tcp,Missing,0.000113,0,0,REJ,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,1
4,192.168.1.193,49180,192.168.1.37,8080,tcp,Missing,0.000130,0,0,REJ,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211038,192.168.1.32,48286,176.28.50.165,80,tcp,http,65.376610,2665,322,S3,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,9
211039,192.168.1.32,48288,176.28.50.165,80,tcp,http,65.710346,1987,322,S3,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,9
211040,192.168.1.32,48292,176.28.50.165,80,tcp,http,65.766512,3922,322,S3,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,9
211041,192.168.1.32,48294,176.28.50.165,80,tcp,http,65.753940,2401,322,S3,...,0,0,Missing,Missing,Missing,Missing,Missing,Missing,1,9


In [88]:
df_new = pd.concat([X, y], axis=1)

In [89]:
# Based on your image: src_ip, src_port, dst_ip, dst_port, proto
df_new['session_id'] = df_new.apply(lambda x: f"{x['src_ip']}:{x['src_port']}-{x['dst_ip']}:{x['dst_port']}-{x['proto']}", axis=1)

# Alternatively, for bidirectional flows (treating A->B and B->A as one session):
def create_bidirectional_id(row):
    sorted_ips = sorted([str(row['src_ip']), str(row['dst_ip'])])
    sorted_ports = sorted([str(row['src_port']), str(row['dst_port'])])
    return f"{sorted_ips[0]}-{sorted_ips[1]}-{sorted_ports[0]}-{sorted_ports[1]}-{row['proto']}"

df_new['bidirectional_session_id'] = df_new.apply(create_bidirectional_id, axis=1)

In [90]:
df_new.columns

Index(['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service',
       'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes',
       'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query',
       'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA',
       'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed',
       'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth',
       'http_method', 'http_uri', 'http_version', 'http_request_body_len',
       'http_response_body_len', 'http_status_code', 'http_user_agent',
       'http_orig_mime_types', 'http_resp_mime_types', 'weird_name',
       'weird_addl', 'weird_notice', 'type', 'session_id',
       'bidirectional_session_id'],
      dtype='object')

## Train Test Split

In [91]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from torch.utils.data import TensorDataset, DataLoader

# --- A. Define Features & Create Sequences ---
feature_cols = [col for col in df_new.columns if col not in 
                ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'proto', 
                 'session_id', 'bidirectional_session_id', 'label', 'type']]

def create_sequences_fixed(df, features, seq_len=64, y_col='type'):
    X, y = [], []
    for _, session in df.groupby('bidirectional_session_id'):
        data = session[features].values
        label = session[y_col].mode()[0]
        if len(data) < seq_len:
            pad = np.zeros((seq_len - len(data), len(features)))
            data = np.vstack([pad, data])
            X.append(data)
            y.append(label)
        else:
            step = seq_len // 2 
            for i in range(0, len(data) - seq_len + 1, step):
                X.append(data[i : i + seq_len])
                y.append(label)
    return np.array(X), np.array(y)

In [92]:
# Encoding labels
df_new['type'] = le.fit_transform(df_new['type'])

In [93]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# STEP 1 & 2: Split the IDs
unique_sessions = df_new['bidirectional_session_id'].unique()
train_ids, test_ids = train_test_split(
    unique_sessions, 
    test_size=0.2, 
    random_state=42,
    shuffle=True
)

# STEP 3: Create train and test dataframes
df_train = df_new[df_new['bidirectional_session_id'].isin(train_ids)]
df_test = df_new[df_new['bidirectional_session_id'].isin(test_ids)]

# STEP 4: Create Sequences (using your fixed feature list)
X_train_seq, y_train_seq = create_sequences_fixed(df_train, feature_cols, seq_len=64)
X_test_seq, y_test_seq = create_sequences_fixed(df_test, feature_cols, seq_len=64)

# STEP 5: Flatten for SMOTE
n_samples, seq_len, n_features = X_train_seq.shape
X_train_flat = X_train_seq.reshape(n_samples, -1)

# STEP 6: Resample Training Data (SMOTE)
sm = SMOTE(random_state=42)
X_resampled_flat, y_resampled_encoded = sm.fit_resample(X_train_flat, y_train_seq)

# --- SCALING STEP ---
# 1. Flatten the Test data to 2D to match the resampled train data shape for the scaler
X_test_flat = X_test_seq.reshape(X_test_seq.shape[0], -1)

# 2. Initialize and apply the Scaler
# We fit ONLY on the Training data to prevent data leakage from the test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_resampled_flat)
X_test_scaled = scaler.transform(X_test_flat)

# --- STEP 7: Reshape back to 3D for the Informer ---
X_train_final = X_train_scaled.reshape(-1, seq_len, n_features)
X_test_final = X_test_scaled.reshape(-1, seq_len, n_features)

# --- STEP 8: Create Tensors & DataLoaders ---
# X_train_final and X_test_final are now scaled and 3D
X_train_tensor = torch.FloatTensor(X_train_final)
y_train_tensor = torch.LongTensor(y_resampled_encoded) 

X_test_tensor = torch.FloatTensor(X_test_final)
y_test_tensor = torch.LongTensor(y_test_seq) # Assuming y_test_seq was already encoded

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor), 
    batch_size=128, 
    shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor), 
    batch_size=128, 
    shuffle=False
)

print(f"Final Train Shape (3D Scaled): {X_train_tensor.shape}")
print(f"Final Test Shape (3D Scaled):  {X_test_tensor.shape}")

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Final Train Shape (3D Scaled): torch.Size([218770, 64, 37])
Final Test Shape (3D Scaled):  torch.Size([24436, 64, 37])


In [94]:
# import torch
# import torch.nn as nn

# class InformerClassifier(nn.Module):
#     def __init__(self, input_dim, seq_len, num_classes, d_model=128, nhead=8, num_layers=3, dropout=0.1):
#         super(InformerClassifier, self).__init__()
        
#         # 1. Feature Projection: Maps your features to the Transformer's d_model space
#         self.embedding = nn.Linear(input_dim, d_model)
        
#         # 2. Learned Positional Encoding: Keeps track of packet order (1 to 64)
#         self.pos_encoder = nn.Parameter(torch.zeros(1, seq_len, d_model))
        
#         # 3. Transformer Encoder: The core "Informer" engine
#         # batch_first=True allows us to use [Batch, Seq, Features]
#         encoder_layers = nn.TransformerEncoderLayer(
#             d_model, nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=True
#         )
#         self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
#         # 4. Global Pooling & Classification Head
#         self.dropout = nn.Dropout(dropout)
#         self.fc = nn.Linear(d_model, num_classes)
        
#     def forward(self, x):
#         # x shape: [Batch, 64, Features]
        
#         # Project and add position
#         x = self.embedding(x) + self.pos_encoder
        
#         # Pass through attention layers
#         x = self.transformer_encoder(x)
        
#         # Average across the 64 packets (Temporal Pooling)
#         x = x.mean(dim=1) 
        
#         x = self.dropout(x)
#         return self.fc(x)


class InformerClassifier(nn.Module):
    def __init__(self, input_dim, seq_len, num_classes, d_model=128, nhead=8, 
                 num_layers=3, dropout=0.1, masking_method='mad', threshold=3.0):
        super(InformerClassifier, self).__init__()
        
        self.masking_method = masking_method
        self.threshold = threshold
        
        # 1. Feature Projection
        self.embedding = nn.Linear(input_dim, d_model)
        
        # 2. Learned Positional Encoding
        self.pos_encoder = nn.Parameter(torch.zeros(1, seq_len, d_model))
        
        # 3. Transformer Encoder
        encoder_layers = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # 4. Classification Head
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def apply_masking(self, x):
        """
        Applies zero-masking to features that exceed the Z-score or MAD threshold.
        Input x shape: [Batch, Seq, d_model]
        """
        if self.masking_method == 'zscore':
            mean = torch.mean(x, dim=1, keepdim=True)
            std = torch.std(x, dim=1, keepdim=True) + 1e-6
            z_scores = torch.abs((x - mean) / std)
            mask = z_scores > self.threshold
            
        elif self.masking_method == 'mad':
            median = torch.median(x, dim=1, keepdim=True).values
            mad = torch.median(torch.abs(x - median), dim=1, keepdim=True).values + 1e-6
            # 0.6745 is the scaling factor to make MAD consistent with Std Dev
            modified_z_score = 0.6745 * torch.abs(x - median) / mad
            mask = modified_z_score > self.threshold
        else:
            return x # No masking applied

        # Zero out the outliers (Masking)
        x = x.masked_fill(mask, 0.0)
        return x

    def forward(self, x):
        # x shape: [Batch, 64, Features]
        
        # Project to d_model space
        x = self.embedding(x)
        
        # --- NEW: Masking Step ---
        # We apply masking in the d_model space to handle latent outliers
        if self.masking_method:
            x = self.apply_masking(x)
        
        # Add position encoding
        x = x + self.pos_encoder
        
        # Pass through attention layers
        x = self.transformer_encoder(x)
        
        # Global Temporal Pooling
        x = x.mean(dim=1) 
        
        x = self.dropout(x)
        return self.fc(x)

In [96]:
# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Initialize the Standalone Informer Model
# input_dim comes from the 3rd dimension of your X_final: [Samples, 64, Features]
input_dim = X.shape 
num_classes = len(class_names)
seq_len = 64

model = InformerClassifier(
    input_dim=input_dim, 
    seq_len=seq_len, 
    num_classes=num_classes,
    d_model=128,    # Model dimension
    nhead=8,        # Number of attention heads
    num_layers=3,   # Number of Transformer layers
    dropout=0.1,
    masking_method="rdmd"
).to(device)
print("Model initialized successfully.")

# 3. Define Loss, Optimizer, and Scheduler
# We use CrossEntropy because it's multiclass (10 categories)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Reduce learning rate when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

TypeError: empty(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got tuple"

## Training

In [55]:
# import time

# epochs = 20
# print(f"Starting training on {device}...")
# print("-" * 30)

# for epoch in range(epochs):
#     start_time = time.time()
    
#     # --- TRAINING PHASE ---
#     model.train()
#     train_loss = 0.0
#     correct_train = 0
#     total_train = 0
    
#     # Use a semicolon here if you want to be extra safe against echoes
#     for batch_X, batch_y in train_loader:
#         batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
#         optimizer.zero_grad()
#         outputs = model(batch_X)
#         loss = criterion(outputs, batch_y)
#         loss.backward()
        
#         # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
#         optimizer.step()
        
#         # .item() is crucial: it converts the tensor to a standard Python float
#         train_loss += loss.item()
        
#         _, predicted = torch.max(outputs.data, 1)
#         total_train += batch_y.size(0)
#         correct_train += (predicted == batch_y).sum().item()

#     # --- VALIDATION PHASE (Added) ---
#     model.eval()
#     val_loss = 0.0
#     correct_val = 0
#     total_val = 0
#     with torch.no_grad():
#         for v_X, v_y in test_loader:
#             v_X, v_y = v_X.to(device), v_y.to(device)
#             v_out = model(v_X)
#             v_l = criterion(v_out, v_y)
#             val_loss += v_l.item()
#             _, v_pred = torch.max(v_out.data, 1)
#             total_val += v_y.size(0)
#             correct_val += (v_pred == v_y).sum().item()

#     # Calculate Averages
#     avg_train_loss = train_loss / len(train_loader)
#     avg_val_loss = val_loss / len(test_loader)
#     train_acc = 100 * correct_train / total_train
#     val_acc = 100 * correct_val / total_val
    
#     # Print clean summary
#     print(f"Epoch [{epoch + 1}/{epochs}] | {time.time() - start_time:.1f}s")
#     print(f"  Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}%")
#     print(f"  Val Loss:   {avg_val_loss:.4f} | Acc: {val_acc:.2f}%")
#     print("-" * 30)

In [57]:
import time
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, 
    roc_auc_score, classification_report
)
import torch.nn.functional as F

epochs = 20
print(f"Starting training on {device}...")
print("-" * 30)

for epoch in range(epochs):
    start_time = time.time()
    
    # Trackers for the current epoch
    train_preds, train_true = [], []
    val_preds, val_true, val_probs = [], [], []
    
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        _ = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
        _, predicted = torch.max(outputs.data, 1)
        train_preds.extend(predicted.cpu().numpy())
        train_true.extend(batch_y.cpu().numpy())

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for v_X, v_y in test_loader:
            v_X, v_y = v_X.to(device), v_y.to(device)
            v_out = model(v_X)
            v_l = criterion(v_out, v_y)
            val_loss += v_l.item()
            
            # Get probabilities for AUROC using Softmax
            probs = F.softmax(v_out, dim=1)
            
            _, v_pred = torch.max(v_out.data, 1)
            val_preds.extend(v_pred.cpu().numpy())
            val_true.extend(v_y.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    # --- COMPREHENSIVE METRICS CALCULATION ---
    
    # 1. Basic Train/Val Accuracy
    epoch_train_acc = accuracy_score(train_true, train_preds) * 100
    epoch_val_acc = accuracy_score(val_true, val_preds) * 100
    
    # 2. Precision, Recall, F1 (Weighted to handle class imbalance in Test set)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_true, val_preds, average='weighted', zero_division=0
    )
    
    # 3. AUROC (Multi-class One-vs-Rest)
    # We use 'ovr' (One-vs-Rest) because we have 10 distinct classes
    try:
        epoch_auroc = roc_auc_score(val_true, val_probs, multi_class='ovr', average='weighted')
    except ValueError:
        epoch_auroc = 0.0 # Handles edge cases where some classes are missing in a batch
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    
    # Output Results
    print(f"Epoch [{epoch + 1}/{epochs}] | {time.time() - start_time:.1f}s")
    print(f"  LOSS:  Train {avg_train_loss:.4f} | Val {avg_val_loss:.4f}")
    print(f"  ACC:   Train {epoch_train_acc:.2f}% | Val {epoch_val_acc:.2f}%")
    print(f"  STATS: Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f} | AUROC: {epoch_auroc:.4f}")
    print("-" * 40)

print("Training Complete!")

Starting training on cuda...
------------------------------


InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

Epoch [1/20] | 13.1s
  LOSS:  Train 0.2174 | Val 0.2198
  ACC:   Train 92.71% | Val 94.53%
  STATS: Prec: 0.9563 | Rec: 0.9453 | F1: 0.9489 | AUROC: 0.9963
----------------------------------------


InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

...

InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

Epoch [19/20] | 13.0s
  LOSS:  Train 0.2088 | Val 0.3560
  ACC:   Train 93.25% | Val 89.90%
  STATS: Prec: 0.9113 | Rec: 0.8990 | F1: 0.9013 | AUROC: 0.9891
----------------------------------------


InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

InformerClassifier(
  (embedding): Linear(in_features=37, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

Epoch [20/20] | 13.0s
  LOSS:  Train 0.2081 | Val 0.2397
  ACC:   Train 93.15% | Val 91.84%
  STATS: Prec: 0.9375 | Rec: 0.9184 | F1: 0.9231 | AUROC: 0.9965
----------------------------------------
Training Complete!


In [58]:
# ==========================================
# 6. Evaluate & Print Metrics
# ==========================================


accuracy = accuracy_score(val_true, val_preds)
print(f"\n---> Test Accuracy: {accuracy * 100:.2f}% <---")
print("\nClassification Report:")
print(classification_report(val_true, val_preds, target_names=class_names))


---> Test Accuracy: 91.84% <---

Classification Report:
              precision    recall  f1-score   support

    backdoor       0.98      0.99      0.99       169
        ddos       0.98      0.95      0.97      3988
         dos       0.62      0.87      0.73       530
   injection       0.96      0.94      0.95      3731
        mitm       0.38      0.86      0.52       186
      normal       0.99      0.92      0.96      5538
    password       0.83      0.96      0.89      3347
  ransomware       0.50      1.00      0.67       316
    scanning       1.00      0.81      0.89      3943
         xss       0.91      0.93      0.92      2688

    accuracy                           0.92     24436
   macro avg       0.82      0.92      0.85     24436
weighted avg       0.94      0.92      0.92     24436

